# Demonstration of ValidatorAgent and ManagerAgent
This notebook tests the new architecture by simulating task execution and validation using a mock labor agent.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from agentlite.agents.ValidatorAgent import ValidatorAgent
from agentlite.agents.ManagerAgent import ManagerAgent
from agentlite.agents.BaseAgent import BaseAgent
from agentlite.commons import TaskPackage
from agentlite.llm.agent_llms import get_llm_backend
from agentlite.llm.LLMConfig import LLMConfig

# Initialize LLM Backend
Set up the LLM backend using LLMConfig and get_llm_backend.

In [2]:
llm_config_dict = {
    "llm_name": "deepseek-r1:8b", # "gemma3:4b"
    "temperature": 0.7, "provider": 
    "ollama", "base_url": 
    "http://localhost:11434"}
llm_config = LLMConfig(llm_config_dict)
llm = get_llm_backend(llm_config)

# Create ValidatorAgent
Instantiate the ValidatorAgent using the initialized LLM backend.

In [3]:
validator_agent = ValidatorAgent(llm)

# Define MockLaborAgent
Define a mock labor agent class that inherits from BaseAgent and provides mock responses for testing.

In [4]:
class MockLaborAgent(BaseAgent):
    def __init__(self, llm):
        super().__init__(
            name="MockLaborAgent",
            role="Provide mock answers for testing.",
            actions=[],
            llm=llm
        )

    def respond(self, task_pkg: TaskPackage, **kwargs):
        return "This is a mock answer to the task."

mock_labor_agent = MockLaborAgent(llm)

# Create ManagerAgent
Instantiate the ManagerAgent with the ValidatorAgent and MockLaborAgent.

In [5]:
manager_agent = ManagerAgent(
    llm=llm,
    name="TestManagerAgent",
    role="Manage tasks and validate responses.",
    TeamAgents=[mock_labor_agent],
    ValidatorAgent=validator_agent
)

# Define and Execute Test Task
Define a test task using TaskPackage, simulate task execution, and print the final response.

In [6]:
test_task = TaskPackage(
    instruction="What is the capital of France?",
    task_creator="Tester",
    task_executor="MockLaborAgent"
)

agent_act = manager_agent.__next_act__(test_task, action_chain=[])
response = manager_agent.forward(test_task, agent_act)

print("Final Response:", response)

Final Response: OK
